### Problem 001: Binary Search (LeetCode 704)

### Problem Definition and Constraints
Given an array of integers `nums` which is sorted in ascending order, and an integer `target`, write a function to search `target` in `nums`. If `target` exists, then return its index. Otherwise, return `-1`. You must write an algorithm with $O(\log n)$ runtime complexity.

* Constraints:
  * 1 <= nums.length <= 10^4
  * -10^4 < nums[i], target < 10^4
  * All integers in `nums` are unique.
  * `nums` is sorted in ascending order.

### Examples
* **Example 1:**
  * Input: `nums = [-1, 0, 2, 4, 6, 8], target = 4`
  * Output: `3`
  * Explanation: 4 exists in nums and its index is 3.
* **Example 2:**
  * Input: `nums = [-1, 0, 2, 4, 6, 8], target = 3`
  * Output: `-1`
  * Explanation: 3 does not exist in nums so return -1.

### Brute Force Approach
The baseline strategy is a standard linear search. We iterate through every element in the array from left to right. If we find the target, we return its index. 
* Time Complexity: $O(n)$ — In the worst-case scenario (the target is at the very end or doesn't exist), we must check every single element.
* Space Complexity: $O(1)$ — No extra memory is used.

### Optimized Approach (Binary Search)
Because the array is strictly sorted, we can use a Binary Search to repeatedly halve our search space. We initialize two pointers: `left` at index 0 and `right` at the last index. In a loop, we calculate the `mid` index. If the value at `mid` matches our target, we return it. If the `mid` value is strictly less than our target, the target must be in the right half, so we update `left = mid + 1`. If the `mid` value is greater, we update `right = mid - 1`.
* Time Complexity: $O(\log n)$ — By eliminating half of the remaining elements on every iteration, the search space shrinks logarithmically. 
* Space Complexity: $O(1)$ — We only maintain three integer pointers (`left`, `right`, `mid`), requiring constant memory.

In [ ]:
from typing import List

class Solution:
    
    # --- BRUTE FORCE APPROACH ---
    def search_brute_force(self, nums: List[int], target: int) -> int:
        for i in range(len(nums)):
            if nums[i] == target:
                return i
        return -1


    # --- OPTIMIZED APPROACH ---
    def search(self, nums: List[int], target: int) -> int:
        # Initialize our boundary pointers
        left = 0
        right = len(nums) - 1
        
        # We loop as long as the search space is valid (pointers haven't crossed)
        while left <= right:
            
            # Calculate the middle index. 
            # Note: (left + right) // 2 works in Python because Python integers don't overflow,
            # but left + ((right - left) // 2) is the universally safe way to write this in C++/Java.
            mid = left + ((right - left) // 2)
            
            if nums[mid] == target:
                # We found the exact target
                return mid
                
            elif nums[mid] < target:
                # The middle value is too small. 
                # We discard the left half by moving the left boundary to mid + 1.
                left = mid + 1
                
            else:
                # The middle value is too large.
                # We discard the right half by moving the right boundary to mid - 1.
                right = mid - 1
                
        # If the loop finishes and we haven't returned, the target isn't in the array
        return -1

### Problem 002: Search a 2D Matrix (LeetCode 74)

### Problem Definition and Constraints
You are given an `m x n` 2D integer array (a matrix) and an integer `target`.
* Each row is sorted in non-decreasing order (smallest to largest).
* The first integer of every row is strictly greater than the last integer of the previous row.
Return `True` if the target exists within the matrix, or `False` otherwise.
* Constraints:
  * 1 <= m, n <= 100
  * -10000 <= matrix[i][j], target <= 10000
  * **Requirement:** Must run in $O(\log(m \cdot n))$ time complexity.

### Examples
* **Example 1:**
  * Input: `matrix = [[1, 2, 4, 8], [10, 11, 12, 13], [14, 20, 30, 40]]`, `target = 10`
  * Output: `True`
* **Example 2:**
  * Input: `matrix = [[1, 2, 4, 8], [10, 11, 12, 13], [14, 20, 30, 40]]`, `target = 15`
  * Output: `False`

### Brute Force Approach
The simplest way to solve this is to ignore the sorted properties entirely and use a nested loop to check every single cell in the matrix one by one until you find the target or run out of cells.
* Time Complexity: $O(m \cdot n)$ — In the worst case, you scan every cell in the `m` rows and `n` columns.
* Space Complexity: $O(1)$ — No extra data structures are used.

### Optimized Approach (Double Binary Search)
Because of the strict sorting rules, the matrix essentially behaves like a single, massive sorted array that was chopped up into rows. We can achieve the required $O(\log(m \cdot n))$ time by running binary search **twice**:
1. **Find the correct row:** We set pointers to the top row and bottom row. We calculate a middle row and check if the target falls within its range (between its first and last values). If the target is smaller than the first value, we discard the bottom half of the rows. If it's larger than the last value, we discard the top half. 
2. **Find the correct column:** Once we isolate the single row where the target *must* be, we run a standard 1D binary search (just like the previous problem) left and right across that specific row.
* Time Complexity: $O(\log m + \log n)$ which mathematically simplifies to $O(\log(m \cdot n))$. Finding the row takes $O(\log m)$ and searching the row takes $O(\log n)$.
* Space Complexity: $O(1)$ — We only store a few integer pointers (`top`, `bot`, `left`, `right`).

In [ ]:
from typing import List

class Solution:
    
    # --- BRUTE FORCE APPROACH ---
    def searchMatrix_brute_force(self, matrix: List[List[int]], target: int) -> bool:
        ROWS = len(matrix)
        COLS = len(matrix[0])
        
        # Linearly scan every cell
        for r in range(ROWS):
            for c in range(COLS):
                if matrix[r][c] == target:
                    return True
                    
        return False

    # --- OPTIMIZED APPROACH (Double Binary Search) ---
    def searchMatrix(self, matrix: List[List[int]], target: int) -> bool:
        ROWS = len(matrix)
        COLS = len(matrix[0])
        
        # Phase 1: Binary Search to find the correct row
        top = 0
        bot = ROWS - 1
        
        while top <= bot:
            row = top + ((bot - top) // 2)
            
            # Check if target is greater than the largest value in this row
            if target > matrix[row][-1]:
                top = row + 1
            # Check if target is smaller than the smallest value in this row
            elif target < matrix[row][0]:
                bot = row - 1
            else:
                # The target is within the range of this row. 
                # Break the loop to lock in our 'row' variable.
                break
                
        # If the loop finished and the pointers crossed, the target doesn't fit in ANY row.
        if not (top <= bot):
            return False
            
        # Phase 2: Standard Binary Search within the locked-in row
        left = 0
        right = COLS - 1
        
        while left <= right:
            mid = left + ((right - left) // 2)
            
            if target > matrix[row][mid]:
                left = mid + 1
            elif target < matrix[row][mid]:
                right = mid - 1
            else:
                # We found the exact target
                return True
                
        return False